# Neural Network untuk Finance: Deteksi Fraud & Skoring Kredit

Referensi: [PyTorch Classification](https://www.learnpytorch.io/02_pytorch_classification/) dan [Nielsen, Bab 1](http://neuralnetworksanddeeplearning.com/chap1.html).

Dua tugas, satu arsitektur, karena keduanya klasifikasi biner (y=1 berarti fraud / gagal bayar):
1. **Fraud**: sangat tidak seimbang (~3% fraud). Akurasi menyesatkan, jadi dipakai PR-AUC dan ROC-AUC.
2. **Credit scoring**: ~20% gagal bayar. Probabilitas default (PD) diubah menjadi skor kredit gaya scorecard.

> Data di sini **sintetis** (`make_classification`) supaya notebook jalan tanpa unduhan. Untuk data nyata (mis. Kaggle *creditcard*, *Give Me Some Credit*), ganti `make_data` dengan pemuatan CSV; sisanya tidak berubah.

## Langkah 1: Import & seed
Seed dikunci agar hasil bisa diulang (sama seperti `random_state=42` di tutorial PyTorch).

In [1]:
import itertools, numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

torch.manual_seed(42)

## Langkah 2: Data, split, dan normalisasi
- **Split 60/20/20 (train/val/test), stratified**: proporsi fraud sama di ketiganya. *Train* untuk belajar, *val* untuk memilih hyperparameter, *test* hanya disentuh sekali di akhir. Jika hyperparameter dipilih memakai test, hasil akhir terlalu optimis.
- **StandardScaler di-fit hanya pada train**: fitur finansial berskala beda (mis. pendapatan vs rasio). Gradient descent jauh lebih stabil pada fitur berskala sama. Fit pada train saja mencegah kebocoran informasi dari val/test.
- Diubah menjadi `torch.float32` tensor, seperti di tutorial.

In [2]:
def make_data(n, n_feat, pos_frac, seed):
    X, y = make_classification(n_samples=n, n_features=n_feat, n_informative=n_feat // 2,
                               n_redundant=2, weights=[1 - pos_frac, pos_frac],
                               flip_y=0.01, random_state=seed)
    Xtr, Xt, ytr, yt = train_test_split(X, y, test_size=0.4, stratify=y, random_state=seed)
    Xva, Xte, yva, yte = train_test_split(Xt, yt, test_size=0.5, stratify=yt, random_state=seed)
    sc = StandardScaler().fit(Xtr)
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    return [(T(sc.transform(a)), T(b)) for a, b in ((Xtr, ytr), (Xva, yva), (Xte, yte))]

fraud = make_data(20000, 20, 0.03, seed=1)
credit = make_data(6000, 10, 0.20, seed=2)
for name, (tr, va, te) in [("fraud", fraud), ("credit", credit)]:
    print(name, "train:", tuple(tr[0].shape), "positif: %.1f%%" % (100 * tr[1].mean()))

fraud train: (12000, 20) positif: 3.5%
credit train: (3600, 10) positif: 20.4%


## Langkah 3: Model (MLP)
Tiap neuron menghitung `w·x + b` lalu fungsi aktivasi (Nielsen bab 1: perceptron → sigmoid neuron). Tutorial PyTorch menekankan bahwa tanpa **non-linearitas (ReLU)** tumpukan layer hanya setara satu garis lurus; dengan ReLU model bisa menangkap pola lengkung.

Arsitektur: `fitur → hidden → ReLU → hidden → ReLU → 1 logit`. Output berupa **logit** (angka mentah); sigmoid baru dipakai saat prediksi menjadi probabilitas.

In [3]:
def build(n_in, hidden):
    return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, 1))

## Langkah 4: Loss, optimizer, dan training loop
- **`BCEWithLogitsLoss`**: sigmoid + binary cross-entropy dalam satu operasi yang stabil secara numerik (disarankan tutorial PyTorch dibanding sigmoid + `BCELoss` terpisah).
- **`pos_weight`**: pada fraud, model yang menebak "semua normal" sudah 97% akurat dan loss-nya kecil. `pos_weight = n_negatif / n_positif` membuat kesalahan pada kelas fraud berbobot lebih besar. Efek sampingnya probabilitas jadi terinflasi, jadi untuk **skoring kredit dimatikan** agar PD tetap terkalibrasi.
- **Mini-batch SGD** (`DataLoader`, Nielsen): gradien dihitung dari potongan kecil data (batch 128), bukan seluruh data, sehingga update lebih sering dan lebih murah. `shuffle=True` mengacak urutan tiap epoch.
- **Adam**: varian gradient descent dengan learning rate adaptif per parameter; lebih toleran daripada SGD polos pada data finansial.
- Loop per batch: `zero_grad` (bersihkan gradien lama) → forward → loss → `backward` (backpropagation, Nielsen bab 2) → `step` (update bobot).

In [5]:
def predict(model, X):
    model.eval()
    with torch.inference_mode():
        return torch.sigmoid(model(X).squeeze(1)).numpy()

def metrics(model, X, y):
    p = predict(model, X)
    return roc_auc_score(y, p), average_precision_score(y, p)

def fit(train, val, hidden=32, lr=1e-2, epochs=20, balance=True, batch=128):
    (Xtr, ytr), (Xva, yva) = train, val
    model = build(Xtr.shape[1], hidden)
    pw = (ytr == 0).sum() / (ytr == 1).sum() if balance else torch.tensor(1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb).squeeze(1), yb).backward()
            opt.step()
    return model, metrics(model, Xva, yva)

## Langkah 5: Tuning hyperparameter (grid search)
Hyperparameter tidak dipelajari oleh gradient descent; kita mencobanya sendiri. Grid 3×3 = 9 percobaan:
- **`hidden`** (8/32/64): kapasitas model. Terlalu kecil → underfit; terlalu besar → overfit dan lambat.
- **`lr`** (0.001/0.01/0.1): ukuran langkah gradient descent (Nielsen: η). Terlalu kecil → lambat/belum konvergen dalam 20 epoch; terlalu besar → loss melompat-lompat.

Pemenang dipilih dari skor **validasi**. Untuk fraud kriteria utamanya PR-AUC (sensitif pada kelas langka); untuk kredit ROC-AUC (kemampuan memeringkat risiko).

In [6]:
def tune(task, key, balance):
    rows = []
    for hidden, lr in itertools.product([8, 32, 64], [1e-3, 1e-2, 1e-1]):
        torch.manual_seed(42)
        _, (roc, pr) = fit(task[0], task[1], hidden=hidden, lr=lr, balance=balance)
        rows.append(dict(hidden=hidden, lr=lr, val_roc_auc=roc, val_pr_auc=pr))
    return pd.DataFrame(rows).sort_values(key, ascending=False).reset_index(drop=True)

fraud_grid = tune(fraud, "val_pr_auc", balance=True)
fraud_grid

,hidden,lr,val_roc_auc,val_pr_auc
0,32,0.010,0.873679,0.650211
1,64,0.010,0.885078,0.647795
2,32,0.001,0.897356,0.646841
3,64,0.001,0.883202,0.643330
4,8,0.001,0.893348,0.632717
5,8,0.010,0.881972,0.629865
6,8,0.100,0.829658,0.626257
7,32,0.100,0.844929,0.609329
8,64,0.100,0.586331,0.201412


In [7]:
credit_grid = tune(credit, "val_roc_auc", balance=False)
credit_grid

,hidden,lr,val_roc_auc,val_pr_auc
0,32,0.010,0.963872,0.902375
1,64,0.010,0.962603,0.916766
2,32,0.100,0.952033,0.884577
3,8,0.100,0.950914,0.860605
4,64,0.100,0.941566,0.870893
5,64,0.001,0.935096,0.838628
6,32,0.001,0.933805,0.817225
7,8,0.010,0.928462,0.821742
8,8,0.001,0.887274,0.666965


## Langkah 6: Evaluasi akhir pada test set (fraud)
Model dilatih ulang dengan hyperparameter terbaik, lalu diuji **sekali** pada test set. Pembanding: tebakan "semua normal" yang akurasinya tinggi tetapi tidak menangkap satu fraud pun.

Karena `pos_weight`, ambang 0.5 cenderung menandai banyak transaksi; di praktik ambang dipilih berdasarkan biaya (kerugian fraud vs biaya memeriksa transaksi sah).

In [8]:
b = fraud_grid.iloc[0]
torch.manual_seed(42)
m_fraud, _ = fit(fraud[0], fraud[1], hidden=int(b.hidden), lr=b.lr, balance=True)
Xte, yte = fraud[2]
print("Baseline 'semua normal' akurasi: %.3f" % (1 - yte.mean()))
print("Test ROC-AUC %.3f | PR-AUC %.3f (PR-AUC acak ≈ %.3f)" % (*metrics(m_fraud, Xte, yte), yte.mean()))
print(classification_report(yte, predict(m_fraud, Xte) > 0.5, target_names=["normal", "fraud"], zero_division=0))

Baseline 'semua normal' akurasi: 0.965
Test ROC-AUC 0.908 | PR-AUC 0.750 (PR-AUC acak ≈ 0.035)
              precision    recall  f1-score   support

      normal       0.99      0.97      0.98      3860
       fraud       0.46      0.75      0.57       140

    accuracy                           0.96      4000
   macro avg       0.72      0.86      0.77      4000
weighted avg       0.97      0.96      0.96      4000



## Langkah 7: Skoring kredit
Model dengan hyperparameter terbaik menghasilkan **PD** (probabilitas gagal bayar). PD diubah menjadi skor gaya scorecard:

`skor = 600 + (20 / ln 2) · ln(odds_good/50)`, dengan `odds_good = (1 − PD) / PD`.

Artinya skor 600 setara odds 50:1 (baik:buruk) dan setiap kenaikan 20 poin menggandakan odds (*points to double the odds*, PDO = 20). Angka 600/50/20 adalah konvensi umum, bukan angka dari data; sesuaikan dengan kebijakan kreditmu.

In [9]:
b = credit_grid.iloc[0]
torch.manual_seed(42)
m_credit, _ = fit(credit[0], credit[1], hidden=int(b.hidden), lr=b.lr, balance=False)
Xte, yte = credit[2]
print("Test ROC-AUC %.3f | PR-AUC %.3f" % metrics(m_credit, Xte, yte))

def credit_score(pd_, base=600, base_odds=50, pdo=20):
    pd_ = np.clip(pd_, 1e-4, 1 - 1e-4)
    return base + pdo / np.log(2) * np.log((1 - pd_) / pd_ / base_odds)

out = pd.DataFrame({"PD": predict(m_credit, Xte), "default": yte.numpy()})
out["skor"] = credit_score(out.PD.values).round()
out["band"] = pd.qcut(out.skor, 5, labels=["E (terburuk)", "D", "C", "B", "A (terbaik)"])
out.groupby("band", observed=True).agg(n=("default", "size"), default_rate=("default", "mean"), skor_rata2=("skor", "mean"))

Test ROC-AUC 0.962 | PR-AUC 0.918


,n,default_rate,skor_rata2
band,,,
E (terburuk),241,0.834025,347.821577
D,240,0.150000,534.166667
C,244,0.008197,602.290984
B,240,0.020833,659.350000
A (terbaik),235,0.004255,726.970213


Tabel di atas adalah pemeriksaan kewajarannya: **default rate harus turun monoton dari band E ke A**. Jika tidak, skor tidak berguna untuk memeringkat risiko.

## Ringkasan keterbatasan
- Data sintetis; angka AUC tidak mewakili performa dunia nyata.
- Grid kecil (9 kombinasi, 1 seed, 20 epoch tetap, tanpa early stopping); selisih kecil antar baris bisa hanya noise.
- Belum ada kalibrasi probabilitas, analisis fairness, atau penjelasan model (SHAP), yang biasanya wajib untuk kredit di dunia nyata.